In [ ]:
import torch
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, KernelPCA
import sys
sys.path.insert(1,'C:\Zmyfiles\codes\research\pyDeepInsight_SRT\pyDeepInsight_SRT\examples\Scripts')


In [ ]:
#import dataset
#SOURCE
expr_file1= r"C:\Zmyfiles\codes\research\KPCA-DA\KPCA-DA\examples\Data\TvsN\TCGA_BRCA_NvT.csv"
expr1 = pd.read_csv(expr_file1, encoding='utf-8', engine='python')
expr1=expr1.transpose()
expr1.columns=expr1.iloc[0]
X1 = expr1.iloc[1:,0:-1].values
X_S=X1.astype(float)
X_S=X_S.T

#TARGET
expr_file2= r"C:\Zmyfiles\codes\research\KPCA-DA\KPCA-DA\examples\Data\TvsN\GSE45498_NvT.csv"
expr2 = pd.read_csv(expr_file2, encoding='utf-8', engine='python')
expr2=expr2.transpose()
expr2.columns=expr2.iloc[0]
X2 = expr2.iloc[1:,0:-1].values
X_T=X2.astype(float)
X_T=X_T.T

In [ ]:
###Normalize the source and target dataset
# scaler = StandardScaler()
# X_S = scaler.fit_transform(X_S)
# X_T = scaler.fit_transform(X_T)
genes1 = expr1.iloc[:,:-1].columns.to_numpy()
genes2 = expr2.iloc[:,:-1].columns.to_numpy()

#extract common genes
# common_indices_X_S = []
# common_indices_X_T = []
# for i, row in enumerate(X_S):
#     for j, candidate in enumerate(X_T):
 
#         if np.array_equal(row, candidate):
#             common_indices_X_S.append(i)
#             common_indices_X_T.append(j)





In [ ]:
#extract common genes
common_genes=[]
for i in genes1:
    for j in genes2:
        if i==j:
            common_genes.append(i)

print(common_genes)


condition1=np.isin(genes1,common_genes)
indexS=np.where(condition1)[0]

condition2=np.isin(genes2,common_genes)
indexT=np.where(condition2)[0]

['ABCB1', 'ABL1', 'AKT1', 'AKT2', 'APC', 'AR', 'AREG', 'ATM', 'BCL2', 'BCL2A1', 'BCL2L1', 'BCL3', 'BCL6', 'BCR', 'BIRC2', 'BIRC5', 'BLM', 'BMI1', 'BRAF', 'BRCA1', 'BRCA2', 'CASP10', 'CASP2', 'CAV1', 'CCNA2', 'CCND1', 'CCND2', 'CCND3', 'CCNE1', 'CD34', 'CD44', 'CDC25B', 'CDC25C', 'CDH1', 'CDH11', 'CDK2', 'CDK4', 'CDK6', 'CDKN1A', 'CDKN2A', 'CDKN2B', 'CDKN2C', 'CEBPA', 'CHEK1', 'CLTC', 'COL1A1', 'CSF1R', 'CSF3R', 'CSK', 'CTNNB1', 'CXCL9', 'CYP1A1', 'DAP3', 'DAPK1', 'DEK', 'DLC1', 'E2F1', 'E2F3', 'EGF', 'EGFR', 'EGR1', 'EPS8', 'ERBB2', 'ERBB3', 'ERBB4', 'ERCC2', 'ERCC4', 'ESR1', 'ETS1', 'ETS2', 'ETV1', 'ETV6', 'FANCG', 'FAS', 'FAT1', 'FGF1', 'FGF2', 'FGFR1', 'FGFR2', 'FGFR3', 'FGFR4', 'FGR', 'FLT1', 'FLT3', 'FOLR1', 'FOS', 'FOSL2', 'FRZB', 'FYN', 'GADD45A', 'GAPDH', 'GAS1', 'GATA1', 'GNAS', 'GRB7', 'GUSB', 'HCK', 'HDAC1', 'HIF1A', 'HMMR', 'HPRT1', 'HRAS', 'HSP90AB1', 'IFNGR1', 'IGF1', 'IGFBP2', 'IGFBP3', 'IGFBP6', 'IL1A', 'IL1B', 'IL4', 'IL6', 'IRF1', 'ITGB1', 'JUN', 'JUNB', 'KDR', 'KIT',

In [ ]:
len(common_genes)

228

In [ ]:
XsShared=X_S[indexS]
XsShared

In [ ]:
# Example data
kpca = KernelPCA(n_components=2, kernel='rbf',gamma=.5)
Z_s = kpca.fit_transform(X_S)
Z_t = kpca.fit_transform(X_T)   # Target domain features
X_t = X_T.T                     # Original target domain data

Z_s = torch.tensor(Z_s, dtype=torch.float32)
Z_t = torch.tensor(Z_t, dtype=torch.float32, requires_grad=True)  # Make Z_t trainable
X_t = torch.tensor(X_t, dtype=torch.float32)

# Extract common points from Z1 and Z2
Zs_common = Z_s[indexS]
Zt_common = Z_t[indexT]

In [ ]:
Zt_common

tensor([[-3.1482e-02, -1.2045e-01],
        [ 4.7408e-01,  4.0927e-02],
        [-7.4458e-03,  8.4504e-02],
        [ 1.4318e-03, -1.2797e-01],
        [-1.1585e-02, -4.6793e-02],
        [ 4.2516e-01,  5.1325e-02],
        [ 1.5385e-02, -9.5517e-03],
        [-1.5286e-02,  5.7640e-01],
        [-1.1429e-01, -1.9963e-01],
        [-1.3374e-02,  3.5365e-02],
        [-1.4235e-01, -1.3347e-01],
        [ 1.1213e-02, -1.0639e-01],
        [-3.5402e-01, -4.6407e-02],
        [ 2.1846e-02, -4.2255e-03],
        [-1.2030e-01,  8.2400e-02],
        [ 1.4003e-02, -2.4259e-01],
        [-2.3517e-02, -5.2272e-02],
        [-1.7671e-02,  2.7111e-02],
        [ 5.6608e-03,  4.1501e-03],
        [-1.8318e-02,  8.3929e-03],
        [ 2.5615e-01, -2.7059e-01],
        [ 6.5588e-03,  6.6679e-03],
        [-1.0161e-01, -4.4531e-02],
        [-1.8065e-02, -2.3021e-02],
        [ 3.4023e-02, -1.3892e-02],
        [ 8.9192e-03, -1.0902e-02],
        [-2.2457e-02, -5.2266e-02],
        [-2.0146e-01,  1.831

In [ ]:
# Define the RBF kernel
def rbf_kernel(x, y, gamma=0.5):
    pairwise_distances = torch.cdist(x, y) ** 2
    return torch.exp(-gamma * pairwise_distances)


In [ ]:
# Define the MMD loss
def mmd_loss(Zs_common, Zt_common, gamma=0.5):
    # if(torch.isnan(Zs_common) or torch.isinf(Zs_common)):
    #     print("NaN Error in Source")
    #     exit()

    # if(torch.isnan(Zt_common) or torch.isinf(Zt_common)):
    #     print("NaN Error in Target")
    #     exit()

    #K_ss = rbf_kernel(Zs_common, Zs_common, gamma)
    K_tt = rbf_kernel(Zt_common, Zt_common, gamma)
    K_st = rbf_kernel(Zs_common, Zt_common, gamma)
    
    #mmd = K_ss.mean() + K_tt.mean() - 2 * K_st.mean()
    mmd = abs( K_tt.mean() - 2 * K_st.mean())
    return mmd


In [ ]:
# Define the regularization term
def regularization_term(Z_t, X_t, gamma=0.5):
    #print(Z_t.shape, X_t.shape)
    K_t = rbf_kernel(X_t.T, X_t.T, gamma)
    #print(K_t.shape)
    A = Z_t.T @ K_t @ Z_t
    trace_A = torch.trace(A)
    return trace_A

In [59]:
# Hyperparameters
lambda_reg = 0.1  # Regularization strength
gamma = .5       # Kernel parameter

# Optimizer
optimizer = optim.SGD([Z_t], lr=0.0001)


In [ ]:
# Training loop
for epoch in range(100):
    
    Z_t = Z_t.detach().requires_grad_(True)
    optimizer.zero_grad()
    
    # Compute MMD loss
    Zt_common = Z_t[indexT]
    mmd = mmd_loss(Zs_common, Zt_common, gamma)
    
    # Compute regularization term
    reg = regularization_term(Z_t, X_t, gamma)
    
    # Total loss
    loss = mmd + lambda_reg * reg
    
    # Backpropagate and update Z_t
    loss.backward(retain_graph=True)
    optimizer.step()
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}, MMD: {mmd.item():.4f}, Reg: {reg.item():.4f}")
    
    #print("Optimized Z_t:", Z_t)
    
# Final Z_t after optimization
#print("Optimized Z_t:", Z_t)

Epoch 0, Loss: 1.1688, MMD: 0.9995, Reg: 1.6925
Epoch 10, Loss: 1.1688, MMD: 0.9995, Reg: 1.6925
Epoch 20, Loss: 1.1688, MMD: 0.9995, Reg: 1.6925
Epoch 30, Loss: 1.1688, MMD: 0.9995, Reg: 1.6925
Epoch 40, Loss: 1.1688, MMD: 0.9995, Reg: 1.6925
Epoch 50, Loss: 1.1688, MMD: 0.9995, Reg: 1.6925
Epoch 60, Loss: 1.1688, MMD: 0.9995, Reg: 1.6925
Epoch 70, Loss: 1.1688, MMD: 0.9995, Reg: 1.6925
Epoch 80, Loss: 1.1688, MMD: 0.9995, Reg: 1.6925
Epoch 90, Loss: 1.1688, MMD: 0.9995, Reg: 1.6925
